# LangChain 의미 검색(Semantic Search) 튜토리얼

이 노트북은 LangChain의 핵심 추상화를 활용하여 PDF 문서에 대한 의미 검색 엔진을 구축하는 과정을 단계별로 안내합니다.

### 핵심 개념
| 요소 | 설명 |
|------|------|
| **문서 로더** | 다양한 소스에서 데이터 수집 |
| **텍스트 분할** | 문서를 관리 가능한 청크로 나누기 |
| **임베딩** | 텍스트를 벡터 표현으로 변환 |
| **벡터 저장소** | 효율적인 유사도 검색을 위해 벡터 저장 |
| **리트리버** | 쿼리와 유사한 문서 검색 |

## 0단계: 환경 설정

`.env` 파일에 `OPENAI_API_KEY`를 설정하고, 필요한 패키지를 import합니다.

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

# API 키가 설정되었는지 확인
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 .env 파일에 설정되지 않았습니다."


## 1단계: Document 객체 이해하기

LangChain의 `Document` 객체는 텍스트 콘텐츠(`page_content`)와 메타데이터(`metadata`)를 포함합니다.

In [16]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companions, known for their loyalty and friendliness.",
        metadata={"source": "mammal-pets-doc"},
    ),
    Document(
        page_content="Cats are independent pets that often enjoy their own space.",
        metadata={"source": "mammal-pets-doc"},
    ),
]

for i, doc in enumerate(documents):
    print(f"문서 {i + 1}:")
    print(f"  내용: {doc.page_content}")
    print(f"  메타데이터: {doc.metadata}\n")

문서 1:
  내용: Dogs are great companions, known for their loyalty and friendliness.
  메타데이터: {'source': 'mammal-pets-doc'}

문서 2:
  내용: Cats are independent pets that often enjoy their own space.
  메타데이터: {'source': 'mammal-pets-doc'}



## 2단계: PDF 문서 로드

`PyPDFLoader`를 사용하여 PDF 파일을 페이지 단위의 `Document` 객체로 로드합니다.

> **참고**: `example_data/` 폴더에 PDF 파일을 넣어주세요.

In [17]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "example_data/nke-10k-2023.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

print(f"로드된 문서(페이지) 수: {len(docs)}")
print(f"\n첫 번째 페이지 내용 (앞 300자):")
print(docs[0].page_content[:300])
print(f"\n첫 번째 페이지 메타데이터: {docs[0].metadata}")

로드된 문서(페이지) 수: 106

첫 번째 페이지 내용 (앞 300자):
FORM 10-KFORM 10-K

첫 번째 페이지 메타데이터: {'producer': 'Wdesk Fidelity Content Translations Version 008.001.016', 'creator': 'Workiva', 'creationdate': '2023-07-20T22:09:22+00:00', 'author': 'anonymous', 'moddate': '2023-07-26T15:13:52+08:00', 'title': 'Nike 2023 Proxy', 'source': 'example_data/nke-10k-2023.pdf', 'total_pages': 106, 'page': 0, 'page_label': '1'}


## 3단계: 텍스트 분할

`RecursiveCharacterTextSplitter`는 공통 구분자를 사용하여 문서를 재귀적으로 분할합니다.

- `chunk_size`: 각 청크의 최대 문자 수
- `chunk_overlap`: 청크 간 겹치는 문자 수 (문맥 유지용)
- `add_start_index`: 원본 문서에서의 시작 위치를 메타데이터에 추가

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"원본 문서 수: {len(docs)}")
print(f"분할 후 청크 수: {len(all_splits)}")
print(f"\n첫 번째 청크 길이: {len(all_splits[0].page_content)}자")
print(f"첫 번째 청크 메타데이터: {all_splits[0].metadata}")
print(f"\n첫 번째 청크 내용:")
print(all_splits[0].page_content)

원본 문서 수: 106
분할 후 청크 수: 501

첫 번째 청크 길이: 18자
첫 번째 청크 메타데이터: {'producer': 'Wdesk Fidelity Content Translations Version 008.001.016', 'creator': 'Workiva', 'creationdate': '2023-07-20T22:09:22+00:00', 'author': 'anonymous', 'moddate': '2023-07-26T15:13:52+08:00', 'title': 'Nike 2023 Proxy', 'source': 'example_data/nke-10k-2023.pdf', 'total_pages': 106, 'page': 0, 'page_label': '1', 'start_index': 0}

첫 번째 청크 내용:
FORM 10-KFORM 10-K


## 4단계: 임베딩

임베딩은 텍스트를 수치 벡터로 변환합니다. 의미가 유사한 텍스트는 벡터 공간에서 가까운 위치에 놓이게 됩니다.

여기서는 OpenAI의 `text-embedding-3-large` 모델을 사용합니다.

In [19]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 임베딩 테스트
sample_text = "LangChain is a framework for building LLM applications."
vector = embeddings.embed_query(sample_text)

print(f"샘플 텍스트: {sample_text}")
print(f"벡터 차원: {len(vector)}")
print(f"벡터 앞 5개 값: {vector[:5]}")

샘플 텍스트: LangChain is a framework for building LLM applications.
벡터 차원: 3072
벡터 앞 5개 값: [-0.04382012411952019, -0.0007473950390703976, -0.039039745926856995, 0.014591953717172146, -0.02463959902524948]


## 5단계: 벡터 저장소 생성

`InMemoryVectorStore`는 메모리 기반의 가벼운 벡터 저장소입니다.
분할된 문서 청크를 임베딩하여 저장합니다.

> **참고**: 프로덕션 환경에서는 Chroma, FAISS, Pinecone 등을 사용할 수 있습니다.

In [20]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)
ids = vector_store.add_documents(documents=all_splits)

print(f"벡터 저장소에 추가된 문서 수: {len(ids)}")

벡터 저장소에 추가된 문서 수: 501


## 6단계: 유사도 검색

벡터 저장소에서 쿼리와 가장 유사한 문서를 검색합니다.

In [21]:
# 기본 유사도 검색
query = "How many distribution centers does Nike have?"  # 나이키는 물류센터가 몇 개인가?
results = vector_store.similarity_search(query)

print(f"쿼리: {query}")
print(f"검색 결과 수: {len(results)}\n")

for i, doc in enumerate(results):
    print(f"--- 결과 {i + 1} ---")
    print(f"내용: {doc.page_content}")
    print(f"메타데이터: {doc.metadata}\n")

쿼리: How many distribution centers does Nike have?
검색 결과 수: 4

--- 결과 1 ---
내용: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 
States are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we 

In [22]:
# 점수 포함 유사도 검색
query = "What was Nike's revenue?"  # 나이키의 매출은 얼마인가?
results_with_score = vector_store.similarity_search_with_score(query)

print(f"쿼리: {query}\n")
for i, (doc, score) in enumerate(results_with_score):
    print(f"결과 {i + 1}: 점수={score:.4f}")
    print(f"  내용: {doc.page_content}\n")

쿼리: What was Nike's revenue?

결과 1: 점수=0.6193
  내용: YEAR ENDED MAY 31,
(Dollars in millions) 2023 2022 2021
REVENUES
North America $ 21,608 $ 18,353 $ 17,179 
Europe, Middle East & Africa  13,418  12,479  11,456 
Greater China  7,248  7,547  8,290 
Asia Pacific & Latin America  6,431  5,955  5,343 
Global Brand Divisions  58  102  25 
Total NIKE Brand  48,763  44,436  42,293 
Converse  2,427  2,346  2,205 
Corporate  27  (72)  40 
TOTAL NIKE, INC. REVENUES $ 51,217 $ 46,710 $ 44,538 
EARNINGS BEFORE INTEREST AND TAXES
North America $ 5,454 $ 5,114 $ 5,089 
Europe, Middle East & Africa  3,531  3,293  2,435 
Greater China  2,283  2,365  3,243 
Asia Pacific & Latin America  1,932  1,896  1,530 
Global Brand Divisions  (4,841)  (4,262)  (3,656) 
Converse  676  669  543 
Corporate  (2,840)  (2,219)  (2,261) 
Interest expense (income), net  (6)  205  262 
TOTAL NIKE, INC. INCOME BEFORE INCOME TAXES $ 6,201 $ 6,651 $ 6,661 
ADDITIONS TO PROPERTY, PLANT AND EQUIPMENT
North America $ 283 $ 146 

In [23]:
# 임베딩 벡터를 직접 생성하여 검색 (embed_query + similarity_search_by_vector)
query = "How were Nike's margins impacted in 2023?"  # 2023년 나이키의 마진은 어떤 영향을 받았는가?
query_vector = embeddings.embed_query(query)
results = vector_store.similarity_search_by_vector(query_vector)

print(f"쿼리: {query}")
print(f"쿼리 벡터 차원: {len(query_vector)}")
print(f"검색 결과 수: {len(results)}\n")

for i, doc in enumerate(results):
    print(f"--- 결과 {i + 1} ---")
    print(f"내용: {doc.page_content}\n")

쿼리: How were Nike's margins impacted in 2023?
쿼리 벡터 차원: 3072
검색 결과 수: 4

--- 결과 1 ---
내용: GROSS MARGIN
FISCAL 2023 COMPARED TO FISCAL 2022
For fiscal 2023, our consolidated gross profit increased 4% to $22,292 million compared to $21,479 million for fiscal 2022. Gross 
margin decreased 250 basis points to 43.5% for fiscal 2023 compared to 46.0% for fiscal 2022 due to the following:
*Wholesale equivalent
The decrease in gross margin for fiscal 2023 was primarily due to:
• Higher NIKE Brand product costs, on a wholesale equivalent basis, primarily due to higher input costs and elevated inbound 
freight and logistics costs as well as product mix;
• Lower margin in our NIKE Direct business, driven by higher promotional activity to liquidate inventory in the current period 
compared to lower promotional activity in the prior period resulting from lower available inventory supply;
• Unfavorable changes in net foreign currency exchange rates, including hedges; and
• Lower off-price margin, on

In [24]:
# 비동기 유사도 검색 (asimilarity_search)
query = "When was Nike incorporated?"  # 나이키는 언제 설립되었는가?
results = await vector_store.asimilarity_search(query)

print(f"쿼리: {query}")
print(f"검색 결과 수: {len(results)}\n")

for i, doc in enumerate(results):
    print(f"--- 결과 {i + 1} ---")
    print(f"내용: {doc.page_content}\n")

쿼리: When was Nike incorporated?
검색 결과 수: 4

--- 결과 1 ---
내용: NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com

--- 결과 2 ---
내용: SDNARBYRAIDISBUS
160 North Washington St.
Boston, Massachusetts 02114
One Bowerman Drive
Beaverton, Oregon 97005-6453
WORLD HEADQUARTERS
One Bowerman Drive
Beaverton, Oregon 97005-6453
EUROPEAN HEADQUARTERS
Colosseum 1
1213 NL Hilversum
The Netherlands
GREATER CHINA HEADQUARTERS
LiNa Building
Tower 1, No. 99
Jiangwancheng Road
Yangpu District
Shanghai, China 200438
SHAREHOLDER INFORMATION
INDEPENDENT ACCOUNTANTS
PricewaterhouseCoopers LLP
805 SW Broadway, Suite 800
Portland, Oregon 97205
REGISTRAR AND STOCK TRANSFER AGENT
Computershare Trust Company, N.A.
P.O. Box 505000
Louisville, KY 40233
800-756-8200
Hearing Impaired #
TDD: 800-952-9245
Shareholder Information
NIKE, Inc. common stock is listed on the New York Stock Exchange under trading symbol ‘NKE.’ Copies of the Company’s Form 10-K or Form
10-Q reports ﬁled with the Securities and Exc

In [25]:
# 리트리버 생성 (similarity 방식)
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

# 단일 쿼리
query = "When was Nike incorporated?"  # 나이키는 언제 설립되었는가?
results = retriever.invoke(query)

print(f"쿼리: {query}")
print(f"검색 결과 수: {len(results)}\n")
for doc in results:
    print(f"내용: {doc.page_content}")

쿼리: When was Nike incorporated?
검색 결과 수: 1

내용: NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com


In [26]:
# 배치 쿼리 - retriever.batch (similarity 방식)
batch_queries = [
    "How many distribution centers does Nike have in the US?",  # 나이키는 미국에 물류센터가 몇 개인가?
    "When was Nike incorporated?",  # 나이키는 언제 설립되었는가?
]
batch_results = retriever.batch(batch_queries)

for query, results in zip(batch_queries, batch_results):
    print(f"쿼리: {query}")
    print(f"  결과: {results[0].page_content}\n")

쿼리: How many distribution centers does Nike have in the US?
  결과: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 
States are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we own.

쿼리: Whe

In [27]:
# score_threshold 직접 구현 (InMemoryVectorStore는 similarity_score_threshold 미지원)
query = "When was Nike incorporated?"  # 나이키는 언제 설립되었는가?
score_threshold = 0.5
k = 1

results_with_score = vector_store.similarity_search_with_score(query, k=k)
# 점수가 threshold 이상인 결과만 필터링
filtered_results = [(doc, score) for doc, score in results_with_score if score >= score_threshold]

print(f"쿼리: {query}")
print(f"score_threshold: {score_threshold}")
print(f"필터링 후 결과 수: {len(filtered_results)}\n")

for doc, score in filtered_results:
    print(f"점수: {score:.4f}")
    print(f"내용: {doc.page_content}")

쿼리: When was Nike incorporated?
score_threshold: 0.5
필터링 후 결과 수: 1

점수: 0.6120
내용: NIKE, INC.
One Bowerman Drive
Beaverton, OR 97005-6453
www.nike.com


In [28]:
# 배치 쿼리 - score_threshold 직접 구현 (for문)
score_threshold = 0.5
k = 1

batch_queries = [
    "How many distribution centers does Nike have in the US?",  # 나이키는 미국에 물류센터가 몇 개인가?
    "When was Nike incorporated?",  # 나이키는 언제 설립되었는가?
]

for query in batch_queries:
    results = vector_store.similarity_search_with_score(query, k=k)
    filtered = [(doc, score) for doc, score in results if score >= score_threshold]

    print(f"쿼리: {query}")
    if filtered:
        doc, score = filtered[0]
        print(f"  점수: {score:.4f} | 결과: {doc.page_content}\n")
    else:
        print(f"  threshold({score_threshold}) 이상 결과 없음\n")

쿼리: How many distribution centers does Nike have in the US?
  점수: 0.7615 | 결과: our Greater China geography, occupied by employees focused on implementing our wholesale, NIKE Direct and merchandising 
strategies in the region, among other functions.
In the United States, NIKE has eight significant distribution centers. Five are located in or near Memphis, Tennessee, two of 
which are owned and three of which are leased. Two other distribution centers, one located in Indianapolis, Indiana and one 
located in Dayton, Tennessee, are leased and operated by third-party logistics providers. One distribution center for Converse is 
located in Ontario, California, which is leased. NIKE has a number of distribution facilities outside the United States, some of 
which are leased and operated by third-party logistics providers. The most significant distribution facilities outside the United 
States are located in Laakdal, Belgium; Taicang, China; Tomisato, Japan and Icheon, Korea, all of which we 